# core

> Gateway naming, startup delivery, and MCP sessions on rustygate gateways

In [ ]:
#| default_exp core

[Rustygate](https://github.com/AnswerDotAI/rustygate) hosts kernels and serves MCP requests at `POST /mcp`. Clikernel runs alongside it for the duration of a conversation. It routes the LLM client's stdio MCP requests to gateways.

This module connects to those gateways. `gateways.toml` gives them names. `session_defaults` supplies kernel startup code and, for a local gateway, the conversation's working directory and environment. It combines `startup.py` and `inspectors.py` into source that can run on another machine.

`Gateway` represents one MCP session on one gateway. `default_gateway` connects to the local gateway or starts a child process if it cannot connect. The caller must stop any returned child. The MCP router does this when the conversation ends.

Rustygate tracks the session's current kernel and which kernels it created. Ending a session stops its autoclose kernels. It leaves other kernels running. Stopping an owned gateway process stops all kernels in that process.


In [ ]:
#| export
import os, tomllib, httpx
from fastcore.utils import *
from fastcore.xdg import xdg_config_home
from mcpmini.core import HTTPTransport, jreq
from rustygate.tools import start_gateway
from clikernel import __version__

In [ ]:
from fastcore.test import *
import asyncio, tempfile
import re


## Configuration

In [ ]:
#| export
DEFAULT_URL = 'http://127.0.0.1:8787'

def cfg_dir():
    "Return the clikernel configuration directory."
    return xdg_config_home()/'clikernel'

def gateways(cfgdir=None):
    "Read named gateways as `{name: {url, token | token_env, verify}}` from `gateways.toml`."
    p = (Path(cfgdir) if cfgdir else cfg_dir())/'gateways.toml'
    return tomllib.loads(p.read_text()).get('gateways', {}) if p.exists() else {}

def resolve(host='', cfgdir=None):
    "Resolve an empty host, URL, or configured gateway name to `(url, token, verify)`."
    if not host: return os.environ.get('CLIKERNEL_HOST', DEFAULT_URL), os.environ.get('CLIKERNEL_TOKEN'), True
    if '://' in host: return host, os.environ.get('CLIKERNEL_TOKEN'), True
    cfgdir = Path(cfgdir) if cfgdir else cfg_dir()
    g = gateways(cfgdir).get(host)
    if g is None: raise ValueError(f"unknown gateway {host!r}: not a URL, and not in {cfgdir/'gateways.toml'}")
    return g['url'], g.get('token') or os.environ.get(g.get('token_env','')) or None, g.get('verify', True)


Clikernel reads its configuration from `$XDG_CONFIG_HOME/clikernel/`, or `~/.config/clikernel/` when the variable is unset. All three files are optional:

- `startup.py` runs in each Python kernel a session creates.
- `inspectors.py` installs functions that check cells before execution.
- `gateways.toml` assigns names and connection settings to gateways.

Use a gateway name to keep authentication tokens out of tool arguments. Tool arguments persist in conversation transcripts. A named gateway can read its token from `token_env` or store it directly as `token`. A nonempty `token` takes precedence.

`resolve` returns `(url, token, verify)`. An empty `host` uses `CLIKERNEL_HOST`, defaulting to `http://127.0.0.1:8787`. A URL passes through unchanged. Both forms use `CLIKERNEL_TOKEN` and enable certificate verification. Any other value names an entry in `gateways.toml`. Unknown names raise `ValueError`.

For example, this entry connects to a gateway using a token from the environment:

    [gateways.solveit]
    url = "https://solveit.example.com/gate"
    token_env = "SOLVEIT_TOKEN"
    verify = false   # accept a self-signed certificate (e.g. rustygate --tls)

`verify = false` disables TLS certificate verification. Omit it to use the default, `true`.


In [ ]:
cfgd = Path(tempfile.mkdtemp())
(cfgd/'gateways.toml').write_text('[gateways.solveit]\nurl = "https://s.example.com/gate"\ntoken = "T"\nverify = false\n')
test_eq(resolve(), (DEFAULT_URL, os.environ.get('CLIKERNEL_TOKEN'), True))
test_eq(resolve('http://h:1/p'), ('http://h:1/p', os.environ.get('CLIKERNEL_TOKEN'), True))
test_eq(resolve('solveit', cfgd), ('https://s.example.com/gate', 'T', False))
test_fail(lambda: resolve('nope', cfgd), contains=str(cfgd/'gateways.toml'))


## Startup and inspectors


Clikernel sends the contents of `startup.py` and `inspectors.py` to the gateway as one Python program. The gateway doesn't need access to those files. This keeps the local configuration in control even when the kernel runs on another machine.

The gateway runs the program before user code in each Python kernel the session creates. `startup.py` runs first. During execution, `__file__` contains its path on the client machine. The wrapper deletes `__file__` afterwards. This preserves the behavior of clikernel v1's `%run -i`. Startup output appears in the reply that announces the new kernel.

The startup program then loads `inspectors.py`. It accepts an `inspectors` list and a function named `inspect`, as in v1. If both exist, the list runs first. Each inspector runs once before each directly submitted cell. A one-argument inspector receives the cell's AST. A two-argument inspector receives `(tree, src)`, including the raw source for lexical checks. Nested execution through a tool such as `%nbrun` skips inspection of the replayed cells.

An inspector can return a note to print before the cell's output, or `None` to say nothing. Raising `RuleBlock` prevents the cell from running. Clikernel supplies this exception in the inspector file's namespace. It subclasses IPython's `InputRejected`, which also blocks execution. Other inspector exceptions print an error but allow the cell to run. An inspector bug must not look like a deliberate policy block.

A failure while loading the file is different from an exception during inspection. Loading errors fail kernel startup. The gateway stops that kernel and returns an error to the creating call. It doesn't leave a kernel running with incomplete inspector setup.

Luau kernels never run this Python startup program or its inspectors. The gateway still applies any working-directory and environment defaults. The gateway's `py`, `lua`, and `create(language=...)` tools control the kernel language. The router doesn't choose it.


In [ ]:
#| export
def _startup_src(src, path):
    "Wrap startup source with `__file__` set to its path during execution and deleted afterwards."
    return f'''__file__ = {str(path)!r}
try: exec(compile({src!r}, __file__, 'exec'))
finally: del __file__'''

In [ ]:
#| export
_INSP_RUNNER = r'''
import inspect as _clik_inspect
import sys as _clik_sys
from IPython.core.error import InputRejected
class RuleBlock(InputRejected):
    "Raise from an inspector to deliberately block a cell; any other inspector exception is a bug, and fails open"

class _ClikInspect:
    "Calls each inspector once per cell: 1-arg get the AST, 2-arg also the raw source"
    def __init__(self, fs): self.fs = fs
    def visit(self, tree):
        fr, n = _clik_sys._getframe(), 0
        while fr:
            n += fr.f_code.co_name == 'run_cell_async'
            fr = fr.f_back
        if n > 1: return tree  # nested run_cell: cell replayed by a tool (%nbrun etc.), not typed
        for f in self.fs:
            try:
                note = f(tree, _clik_src) if len(_clik_inspect.signature(f).parameters) > 1 else f(tree)
                if note: print(note, end='')
            except InputRejected: raise
            except Exception as e: print(f'inspector error (cell runs anyway): {e!r}')
        return tree

def _clik_stash(info):
    global _clik_src
    _clik_src = info.raw_cell

def _clik_install(src):
    ns = dict(RuleBlock=RuleBlock)
    exec(compile(src, 'inspectors.py', 'exec'), ns)
    fs = list(ns.get('inspectors') or [])
    if callable(ns.get('inspect')): fs.append(ns['inspect'])
    if fs:
        ip = get_ipython()
        ip.events.register('pre_run_cell', _clik_stash)
        ip.ast_transformers.append(_ClikInspect(fs))
_clik_src = ''
'''

def _inspector_setup(src):
    "Kernel-side source installing the inspectors defined in `src`; a load failure raises, failing the create call"
    return _INSP_RUNNER + f'\n_clik_install({src!r})'

`startup_src` returns the startup program, with `startup.py` before the inspector installer. `session_defaults` includes that source in the gateway's initialization settings:

In [ ]:
#| export
def startup_src(cfgdir=None):
    "Combine wrapped `startup.py` source with the inspector installer, in that order."
    d = Path(cfgdir) if cfgdir else cfg_dir()
    parts = []
    if (p := d/'startup.py').exists(): parts.append(_startup_src(p.read_text(), p))
    if (p := d/'inspectors.py').exists(): parts.append(_inspector_setup(p.read_text()))
    return '\n'.join(parts)

def session_defaults(cfgdir=None, quiet=False, local=True):
    "Return startup and quiet settings for rustygate initialization. Local sessions also include cwd and env."
    d = dict(startup=startup_src(cfgdir), quiet=quiet)
    if local:
        d['cwd'] = os.getcwd()
        d['env'] = dict(os.environ, CLIKERNEL_QUIET='1') if quiet else dict(os.environ)
    return d

`session_defaults` returns the fields for the `rustygate` extension to MCP initialization. It always includes `startup` and `quiet`. With `local=True`, it also includes the current working directory and a copy of the environment.

The router uses `local=False` for named hosts. This avoids sending local paths and environment variables to a different machine. Startup source still goes to that gateway.

For local kernels, `quiet=True` adds `CLIKERNEL_QUIET=1` to the environment. Kernel-side tools can use it to suppress their own messages. For example, llmdojo uses it for rule warnings.

In [ ]:
cfgd = Path(tempfile.mkdtemp())
(cfgd/'startup.py').write_text('import sys\nbase = 42\nprint("ready, file", __file__.rsplit("/",1)[-1])')
insp_src = '''
import ast
def note_sleep(tree, src):
    if 'time.sleep' in src: return 'note: sleeping\\n'
def block_ctypes(tree):
    if any(isinstance(n, ast.Import) and any(a.name=='ctypes' for a in n.names) for n in ast.walk(tree)):
        raise RuleBlock('no ctypes in kernels')
def buggy(tree, src):
    if 'trigger_bug' in src: raise TypeError('oops')
inspectors = [note_sleep, block_ctypes, buggy]
'''
(cfgd/'inspectors.py').write_text(insp_src)
d = session_defaults(cfgd)
assert 'ready, file' in d['startup'] and '_clik_install' in d['startup']
test_eq(session_defaults(cfgd, local=False).keys(), {'startup', 'quiet'})
sorted(d)

['cwd', 'env', 'quiet', 'startup']

## The gateway session

`Gateway` opens an MCP session with `initialize`, sends tool calls, and ends the session with an HTTP DELETE. Rustygate tracks the current kernel and autoclose policy. The client class manages the connection and request ids, not kernel ownership.

`aclose` closes the HTTP connection even if the termination request raises. It propagates the error after local cleanup. A failed request does not establish that the remote kernels stopped.

Authentication uses a bearer token. `verify=False` disables certificate verification, including for self-signed certificates.

`text` joins a tool reply's text blocks into one string. It raises `RuntimeError` when the reply has `isError`. Use `call` when you need the full reply, including non-text content.

In [ ]:
#| export
class Gateway:
    "Open an MCP session on rustygate, call its tools, and end the session with DELETE."
    def __init__(self,
        url,          # The gateway base URL, e.g. 'http://127.0.0.1:8787'
        token=None,   # Gateway auth token, sent as a bearer token
        verify=True,  # Verify TLS certificates?
    ):
        client = httpx.AsyncClient(verify=verify, timeout=httpx.Timeout(None, connect=10))
        self.url,self._id = url,0
        self.tr = HTTPTransport(f"{url.rstrip('/')}/mcp", token=token, http_client=client)

    async def rpc(self, method, **params):
        "Send a JSON-RPC request and return its result. Raise `RuntimeError` for protocol errors."
        self._id += 1
        r = await self.tr.send(jreq(method, self._id, **params))
        if 'error' in r: raise RuntimeError(f"{r['error']['code']}: {r['error']['message']}")
        return r['result']

    async def initialize(self, defaults=None):
        "Open the MCP session with `defaults` in the `rustygate` extension. Return self."
        await self.tr.start()
        self.info = await self.rpc('initialize', protocolVersion='2025-11-25', capabilities={},
            clientInfo=dict(name='clikernel', version=__version__), rustygate=defaults or {})
        self.tr.proto = self.info['protocolVersion']
        await self.tr.send(jreq('notifications/initialized'))
        return self

    async def tools(self): return (await self.rpc('tools/list'))['tools']
    async def call(self, name, **args): return await self.rpc('tools/call', name=name, arguments=args)

    async def text(self, name, **args):
        "Join a tool reply's text blocks. Raise `RuntimeError` for `isError` replies."
        r = await self.call(name, **args)
        t = ''.join(c.get('text','') for c in r['content'] if c['type'] == 'text')
        if r.get('isError'): raise RuntimeError(t)
        return t

    async def aclose(self):
        "Request session termination and close the HTTP connection."
        try: await self.tr.delete()
        finally: await self.tr.aclose()

Let's start a disposable rustygate process on a free local port. We'll connect using the temporary configuration above.

The first `py` call has no current kernel. Rustygate creates one and runs the startup program. The reply contains its kernel id and the startup banner. Even an empty Python cell produces this startup output:

In [ ]:
g = start_gateway()
os.environ['CLIK_DEMO'] = 'via-defaults'
gw = await Gateway(g.url).initialize(session_defaults(cfgd))
banner = await gw.text('py', code='')
kid = re.search(r'created kernel (\w+)', banner).group(1)
assert 'ready, file startup.py' in banner
banner

'created kernel 20c8121b9c7a42bd88155ad707ba1445 language=python\nready, file startup.py\n'

The next calls reuse the kernel's state. Its working directory matches the conversation's. Its environment includes `CLIK_DEMO`, which we set after starting the gateway. That value came from the session defaults, not the gateway process's inherited environment.

IPython magics also work. Run shell code with `%%bash` through `py` rather than a separate tool:


In [ ]:
test_eq(await gw.text('py', code='base'), '42')
test_eq(await gw.text('py', code='import os; os.getcwd()'), repr(os.getcwd()))
test_eq(await gw.text('py', code="os.environ['CLIK_DEMO']"), "'via-defaults'")
hi = await gw.text('py', code='%%bash\necho hi')
test_eq(hi, 'hi\n')
hi


'hi\n'

The inspectors above handle these cells differently. Sleeping prints a note before the result. The deliberate inspector bug prints an error but still permits execution. Importing `ctypes` raises `RuleBlock` before the cell runs. The kernel retains its earlier state:


In [ ]:
noted = await gw.text('py', code='import time; time.sleep(0.01); 7')
assert noted.startswith('<stdout>\nnote: sleeping') and '7' in noted
buggy = await gw.text('py', code='trigger_bug = 1; 8')
assert 'inspector error (cell runs anyway)' in buggy and '8' in buggy
blocked = await gw.text('py', code='import ctypes')
assert 'no ctypes in kernels' in blocked
test_eq(await gw.text('py', code='base'), '42')
blocked

"---------------------------------------------------------------------------\nRuleBlock                                 Traceback (most recent call last)\nCell In[1], line 24, in _ClikInspect.visit(self, tree)\n     20         for f in self.fs:\n     21             try:\n     22                 note = f(tree, _clik_src) if len(_clik_inspect.signature(f).parameters) > 1 else f(tree)\n     23                 if note: print(note, end='')\n---> 24             except InputRejected: raise\n     25             except Exception as e: print(f'inspector error (cell runs anyway): {e!r}')\n     26         return tree\n\nFile inspectors.py:7, in block_ctypes(tree)\n      5 'Could not get source, probably due dynamically evaluated source code.'\n\nRuleBlock: no ctypes in kernels"

`create` binds a kernel to a dialog name. A new kernel closes with its creating session by default. Pass `autoclose=False` to keep it running after that session ends.

`use_kernel` selects an existing kernel without taking responsibility for its lifetime. Here, a second session selects the first session's automatic Python kernel. Closing the first session stops that kernel even though the second session uses it. The named `demo.ipynb` kernel also stops. Only the kernel created with `autoclose=False` remains:

In [ ]:
assert (await gw.text('create', dlgname='demo.ipynb')).startswith('created kernel ')
kept = re.search(r'kernel (\w+)', await gw.text('create', dlgname='keeper.ipynb', autoclose=False)).group(1)

gw2 = await Gateway(g.url).initialize(session_defaults(cfgd))
assert kid[:8] in await gw2.text('use_kernel', kernel=kid[:8])
test_eq(await gw2.text('py', code='base'), '42')

await gw.aclose()
listing = await gw2.text('list_kernels')
assert kid not in listing and 'demo.ipynb' not in listing and kept in listing
listing

'74b4b067f72b45c0bda293baa1f12483  alive  language=python  connections=0  dlgname=keeper.ipynb'

`restart` replaces the interpreter but keeps the kernel id. If this session created the kernel, rustygate also reruns its startup program. Here, startup prints the banner and restores `base`. The later assignment to `y` does not survive.

The next example gives a new session an `inspectors.py` file that cannot load. Creating a kernel fails with `startup failed`. The failed kernel does not remain in the gateway's kernel list.

In [ ]:
assert 'created kernel' in await gw2.text('create', dlgname='r.ipynb')
await gw2.text('py', code='y = 5')
res = await gw2.text('restart')
assert res.startswith('restarted kernel ') and 'ready' in res
test_eq(await gw2.text('py', code='base'), '42')
assert 'NameError' in await gw2.text('py', code='y')
res

'restarted kernel 4ff6fff201d04929a2d512e9da6ecc6cready, file startup.py\n'

In [ ]:
bad = Path(tempfile.mkdtemp())
(bad/'inspectors.py').write_text('import not_a_module')
gw3 = await Gateway(g.url).initialize(session_defaults(bad))
before = await gw2.text('list_kernels')
with expect_fail(RuntimeError, contains='startup failed'): await gw3.text('py', code='1')
after = await gw2.text('list_kernels')
test_eq(after, before)
after

'4ff6fff201d04929a2d512e9da6ecc6c  alive  language=python  connections=1  dlgname=r.ipynb  <- current\n74b4b067f72b45c0bda293baa1f12483  alive  language=python  connections=0  dlgname=keeper.ipynb'

`quiet=True` suppresses startup output without skipping startup. The reply still identifies the new kernel. For this local session, the defaults also set `CLIKERNEL_QUIET=1` for kernel-side tools. `base` remains available even though the reply no longer contains the banner:

In [ ]:
gwq = await Gateway(g.url).initialize(session_defaults(cfgd, quiet=True))
qbanner = await gwq.text('py', code='')
assert 'created kernel' in qbanner and 'ready' not in qbanner
test_eq(await gwq.text('py', code='base'), '42')
test_eq(await gwq.text('py', code="import os; os.environ['CLIKERNEL_QUIET']"), "'1'")
await gwq.aclose()
qbanner

'created kernel dcccea48eef544a1a743b174afd87be6 language=python\n'

## The default gateway

`default_gateway` tries to initialize a session at the default local URL. If connecting raises `httpx.ConnectError`, it starts a rustygate child on a free port and connects there. It returns the initialized `Gateway` and the child process, or `None` instead of a child when it reused a running gateway.

The caller must close the session and stop any returned child. The MCP router does this when the conversation ends. It doesn't stop a gateway that was already running.

A kernel can outlive the conversation only if its gateway also stays running. Start a separate `rustygate` service when you need persistence. Kernels created with autoclose still stop at session end, even on that service. Use `autoclose=False` to keep a newly created kernel. Nothing survives shutdown of a child gateway.

In [ ]:
#| export
async def default_gateway(
    cfgdir=None,  # Config dir for `session_defaults` (the standard one if None)
    quiet=False,  # Keep startup output out of replies?
):
    "Return an initialized `Gateway` and its new child process, or None if it reused a gateway. The caller must stop any child."
    url, token, verify = resolve('', cfgdir)
    d = session_defaults(cfgdir, quiet)
    try: return await Gateway(url, token, verify).initialize(d), None
    except httpx.ConnectError:
        child = start_gateway()
        return await Gateway(child.url).initialize(d), child

First, point `CLIKERNEL_HOST` at our running test gateway. `default_gateway` connects without starting another process:


In [ ]:
os.environ['CLIKERNEL_HOST'] = g.url
gf, child = await default_gateway(cfgd)
assert child is None and gf.url == g.url
await gf.aclose()
gf.url


'http://127.0.0.1:55260'

Now point it at a port with no listener. `default_gateway` starts a child and applies the same startup settings. We close the session and stop that child after checking the reply:

In [ ]:
os.environ['CLIKERNEL_HOST'] = 'http://127.0.0.1:1'
go, child = await default_gateway(cfgd)
assert child is not None and go.url == child.url
banner = await go.text('py', code='')
assert 'created kernel' in banner and 'ready' in banner
await go.aclose()
child.stop()
child.url

'http://127.0.0.1:55348'

In [ ]:
#|hide
await gw2.aclose()
g.stop()
for k in ('CLIKERNEL_HOST', 'CLIK_DEMO'): os.environ.pop(k, None)

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()